# MDCE Real Data Pipeline For Google Colab

Run this notebook inside Google Colab. Codex cannot directly log in to your Google account or control Colab, but this notebook gives you the exact workflow to run with Google Drive.

It will:

- mount Google Drive
- create the MDCE Drive folders
- download the small public dataset if it is not already present
- extract the parquet file
- prepare an MDCE-ready CSV
- run the confidence engine
- save outputs back to Drive

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Configure Paths

Put this project folder in Drive at `MyDrive/MDCE/project`, or change `PROJECT_DIR` below to wherever your repo/code folder is.

In [ ]:
from pathlib import Path
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/MDCE')
PROJECT_DIR = DRIVE_ROOT / 'project'
RAW_DIR = DRIVE_ROOT / 'data' / 'raw'
EXTRACTED_DIR = RAW_DIR / 'extracted'
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / 'reports'

for folder in [RAW_DIR, EXTRACTED_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not (PROJECT_DIR / 'src').exists():
    raise RuntimeError(
        f'MDCE project code not found at {PROJECT_DIR}. Upload this project folder there, or change PROJECT_DIR.'
    )

sys.path.insert(0, str(PROJECT_DIR))
print('Project:', PROJECT_DIR)
print('Raw data:', RAW_DIR)
print('Processed data:', PROCESSED_DIR)

## 3. Install Runtime Dependencies

In [ ]:
%pip install -q pandas pyarrow numpy plotly requests

## 4. Download The Small Weather/Tyre Dataset

This uses the public Kaggle dataset download endpoint. If it fails in Colab, manually download the zip from Kaggle and upload it to `MDCE/data/raw/` in Drive.

In [ ]:
import requests

DATASET_URL = 'https://www.kaggle.com/api/v1/datasets/download/navenkumar1998/formula-1-dataset-with-weather-and-tyre-features'
ZIP_PATH = RAW_DIR / 'kaggle-naven-weather-tyre-features.zip'

if ZIP_PATH.exists():
    print('Already exists:', ZIP_PATH)
else:
    with requests.get(DATASET_URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        with ZIP_PATH.open('wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
    print('Downloaded:', ZIP_PATH, ZIP_PATH.stat().st_size, 'bytes')

## 5. Extract The Parquet File

In [ ]:
import zipfile

TARGET_EXTRACT_DIR = EXTRACTED_DIR / 'kaggle_naven_weather_tyre'
TARGET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = TARGET_EXTRACT_DIR / 'f1_all.parquet'

if PARQUET_PATH.exists():
    print('Already extracted:', PARQUET_PATH)
else:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(TARGET_EXTRACT_DIR)
    print('Extracted:', PARQUET_PATH)

## 6. Prepare MDCE CSV

Default behavior selects the latest race in the dataset, chooses the best driver by available laps and finish position, and cuts the data at the first detected stint-change decision lap.

In [ ]:
from src.dataset_adapters import prepare_kaggle_weather_tyre

prepared = prepare_kaggle_weather_tyre(PARQUET_PATH)
PROCESSED_CSV = PROCESSED_DIR / 'mdce_kaggle_weather_tyre_latest.csv'
METADATA_JSON = PROCESSED_DIR / 'mdce_kaggle_weather_tyre_latest.metadata.json'

prepared.frame.to_csv(PROCESSED_CSV, index=False)
METADATA_JSON.write_text(__import__('json').dumps(prepared.metadata, indent=2), encoding='utf-8')

print('Wrote:', PROCESSED_CSV)
print('Wrote:', METADATA_JSON)
prepared.metadata

## 7. Validate Through MDCE Loader

In [ ]:
from src.data_loader import load_race_csv

data_load = load_race_csv(PROCESSED_CSV, source_name=str(PROCESSED_CSV))
print('Records:', len(data_load.records))
print('Real/source columns:', data_load.real_columns)
print('Derived columns:', data_load.derived_columns)
print('Proxy columns:', data_load.proxy_columns)
for warning in data_load.warnings[:12]:
    print('-', warning)

## 8. Run Confidence Analysis

In [ ]:
from src.models import ScenarioFlags
from src.pipeline import analyze_decision

result, scenario_records, scenario_notes, conflict = analyze_decision(
    data_load.records,
    ScenarioFlags(),
    prefer_granite=False,
)

summary = {
    'recommendation': result.recommendation.recommendation_type.value,
    'recommended_lap': result.recommendation.recommended_lap,
    'confidence': round(result.confidence.confidence, 3),
    'risk_level': result.confidence.risk_level,
    'conflict_score': conflict[0],
    'conflict_label': conflict[1],
    'issues': [issue.issue for issue in result.issues],
    'fallback_actions': result.fallback_actions,
    'explanation': result.explanation,
}
summary

## 9. Compare Failure Scenarios

In [ ]:
import pandas as pd

scenarios = {
    'normal': ScenarioFlags(),
    'missing_telemetry': ScenarioFlags(missing_telemetry=True),
    'model_mismatch': ScenarioFlags(model_mismatch=True),
    'tyre_signal_drift': ScenarioFlags(tyre_signal_drift=True),
    'safety_car_phase': ScenarioFlags(safety_car_phase=True),
    'weather_uncertainty': ScenarioFlags(weather_uncertainty=True),
}

rows = []
for name, flags in scenarios.items():
    scenario_result, _, _, scenario_conflict = analyze_decision(data_load.records, flags, prefer_granite=False)
    rows.append({
        'scenario': name,
        'recommendation': scenario_result.recommendation.recommendation_type.value,
        'lap': scenario_result.recommendation.recommended_lap,
        'confidence': round(scenario_result.confidence.confidence, 3),
        'risk': scenario_result.confidence.risk_level,
        'conflict': scenario_conflict[1],
        'issues': ', '.join(issue.issue for issue in scenario_result.issues),
    })

comparison = pd.DataFrame(rows)
comparison

## 10. Save Report Output To Drive

In [ ]:
import json

REPORT_PATH = OUTPUT_DIR / 'mdce_colab_real_data_report.json'
REPORT_PATH.write_text(
    json.dumps({'metadata': prepared.metadata, 'summary': summary, 'scenario_comparison': comparison.to_dict(orient='records')}, indent=2),
    encoding='utf-8',
)
print('Saved:', REPORT_PATH)

## Team Workflow

Your friend can open the same notebook from the shared Drive folder. One person should own data preparation, and one person should own app/code changes to avoid overwriting each other.